<a href="https://colab.research.google.com/github/GillValenzuela/curso_data_science/blob/master/DS_Ingemat_Clase_23.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1) Uninstall any old torch installs
!pip uninstall -y torch torchvision torchaudio

# 2) Upgrade pip
!pip install --upgrade pip --quiet

# 3) Install the CUDA-12.4 PyTorch wheels, but skip their nvidia-* deps
!pip install --extra-index-url https://download.pytorch.org/whl/cu124 \
    torch==2.6.0+cu124 \
    torchvision==0.21.0+cu124 \
    torchaudio==2.6.0+cu124 \
    --no-deps --quiet

# 4) Reinstall the HF stack and pin fsspec/gcsfs so load_dataset("imdb") works
!pip install \
    datasets \
    huggingface_hub \
    fsspec==2025.3.0 \
    gcsfs==2025.3.0 --quiet

Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124
Found existing installation: torchaudio 2.6.0+cu124
Uninstalling torchaudio-2.6.0+cu124:
  Successfully uninstalled torchaudio-2.6.0+cu124
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [torch]


In [3]:
# ---------------------------------------------------------------
# DEMO: One-Hot   vs.   Embedding  (PyTorch)
# ---------------------------------------------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

In [4]:
# 1. Vocabulario mínimo
vocab = {"<pad>": 0, "<unk>": 1, "i": 2, "love": 3, "like": 4, "pytorch": 5}
V = len(vocab)          # 6 tokens
d = 8                   # dimensión del embedding (pequeña para demo)

# 2. Frase de ejemplo
sentence = ["i", "love", "pytorch"]
idxs = torch.tensor([vocab[w] for w in sentence])          # [2, 3, 5]

# 3. ONE-HOT (sparse)
one_hot = F.one_hot(idxs, num_classes=V).float()           # (3, 6)

# 4. EMBEDDING (denso)
torch.manual_seed(0)                                       # reproducible
embed = nn.Embedding(num_embeddings=V, embedding_dim=d, padding_idx=0)
vecs = embed(idxs)                                         # (3, 8)

# 5. Similitud coseno entre "love" y "like"
love_vec = embed(torch.tensor([vocab["love"]]))            # (1, 8)
like_vec = embed(torch.tensor([vocab["like"]]))
cos_sim  = F.cosine_similarity(love_vec, like_vec).item()

# 6. Mostrar resultados
print("=== ONE-HOT (forma):", one_hot.shape)
print(one_hot, "\n")

print("=== EMBEDDING vectors (forma):", vecs.shape)
print(pd.DataFrame(vecs.detach().numpy(),
                   index=sentence, columns=[f"d{i}" for i in range(d)]), "\n")

print("Similitud coseno  <love, like>  =", round(cos_sim, 3))

=== ONE-HOT (forma): torch.Size([3, 6])
tensor([[0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 0., 1.]]) 

=== EMBEDDING vectors (forma): torch.Size([3, 8])
               d0        d1        d2        d3        d4        d5        d6  \
i       -1.352654 -1.695931  0.566651  0.793508  0.598839 -1.555095 -0.341360   
love     0.750189 -0.585498 -0.173397  0.183478  1.389366  1.586334  0.946298   
pytorch -0.102310  0.792444 -0.289668  0.052507  0.522860  2.302205 -1.468894   

               d7  
i        1.853006  
love    -0.843677  
pytorch -1.586689   

Similitud coseno  <love, like>  = 0.209


In [5]:
!pip install scikit-learn tqdm --quiet

In [6]:
!pip install --upgrade \
    fsspec==2025.3.0 \
    gcsfs==2025.3.0

  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: gcsfs
    Found existing installation: gcsfs 2025.3.2
    Uninstalling gcsfs-2025.3.2:
      Successfully uninstalled gcsfs-2025.3.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gcsfs]


In [6]:
# ==============================================================
# BOW + MLP  •  IMDB sentiment  •  PyTorch + scikit-learn
# ==============================================================
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import Adam
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm
from datasets import load_dataset
from torch.utils.data import DataLoader

In [7]:
# --------------------------------------------------------------
# 1) Cargar dataset IMDB (25 000 train + 25 000 test)
# --------------------------------------------------------------
raw        = load_dataset("imdb")
train_ds   = raw["train"]
test_ds    = raw["test"]


def load_IMDB(dataset):
    # dataset is a datasets.Dataset, with fields "text" and "label"
    texts  = dataset["text"]   # a list of all examples' texts
    labels = dataset["label"]  # a list of 0/1 labels
    return texts, labels

texts_train, labels_train = load_IMDB(train_ds)
texts_test,  labels_test  = load_IMDB(test_ds)


In [8]:
texts_train[0]

'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, ev

In [9]:
# --------------------------------------------------------------
# 2) TF-IDF (1-gram + 2-gram)  •  max 20 000 features
# --------------------------------------------------------------
vectorizer = TfidfVectorizer(max_features=20_000,
                             ngram_range=(1, 2),
                             stop_words="english")

X_train = vectorizer.fit_transform(texts_train).astype("float32")
X_test  = vectorizer.transform(texts_test).astype("float32")

In [10]:
# --------------------------------------------------------------
# 3) Convertir a tensores  + DataLoaders
# --------------------------------------------------------------
y_train = torch.tensor(labels_train, dtype=torch.long)
y_test  = torch.tensor(labels_test,  dtype=torch.long)

X_train_t = torch.tensor(X_train.toarray())
X_test_t  = torch.tensor(X_test.toarray())

train_ds = TensorDataset(X_train_t, y_train)
test_ds  = TensorDataset(X_test_t,  y_test)

train_dl = DataLoader(train_ds, batch_size=256, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=512)

In [11]:
# --------------------------------------------------------------
# 4) Definir MLP
# --------------------------------------------------------------
input_dim = X_train_t.shape[1]     # = 20 000
model = nn.Sequential(
    nn.Linear(input_dim, 256),
    nn.ReLU(),
    nn.Linear(256, 2)              # 2 clases: neg / pos
)

In [12]:
# --------------------------------------------------------------
# 5) Entrenar
# --------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=1e-3)

epochs = 3
for ep in range(1, epochs + 1):
    model.train()
    for xb, yb in tqdm(train_dl, desc=f"Epoch {ep}", leave=False):
        xb, yb = xb.to(device), yb.to(device)
        preds = model(xb)
        loss  = criterion(preds, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # validación rápida
    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for xb, yb in test_dl:
            preds = model(xb.to(device)).argmax(1).cpu()
            all_preds.extend(preds.numpy())
            all_true.extend(yb.numpy())
    acc = accuracy_score(all_true, all_preds)
    print(f"Epoch {ep}: test accuracy = {acc*100:.2f}%")

# --------------------------------------------------------------
# 6) Guardar modelo (opcional)
# --------------------------------------------------------------
torch.save(model.state_dict(), "bow_mlp_imdb.pt")

Epoch 1:   0%|          | 0/98 [00:00<?, ?it/s]

Epoch 1: test accuracy = 88.67%


Epoch 2:   0%|          | 0/98 [00:00<?, ?it/s]

Epoch 2: test accuracy = 87.90%


Epoch 3:   0%|          | 0/98 [00:00<?, ?it/s]

Epoch 3: test accuracy = 87.00%


In [1]:
# ===============================================================
# Bi-LSTM • IMDB Sentiment Classification • PyTorch ≥ 2.0
# ===============================================================

import re
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from tqdm.auto import tqdm
from datasets import load_dataset
from collections import Counter

# --------------------------------------------------------------
# 0) Cargar IMDB (25 000 train + 25 000 test)
# --------------------------------------------------------------
raw      = load_dataset("imdb")
train_ds = raw["train"]
test_ds  = raw["test"]

# ---------------------------------------------------------------
# “basic_english” tokenizer
# ---------------------------------------------------------------
def basic_english_tokenizer(text: str):
    return re.findall(r"\w+|[^\s\w]+", text.lower())

# ---------------------------------------------------------------
# 2) Build vocab from training set
# ---------------------------------------------------------------
def build_vocab_from_iterator(iterator, specials=("<pad>","<unk>")):
    freq = Counter(tok for tokens in iterator for tok in tokens)
    tokens = list(specials) + [w for w,_ in freq.most_common()]
    vocab = {w:i for i,w in enumerate(tokens)}
    return vocab, vocab["<unk>"]

# stream of token lists over the training texts
token_stream = (basic_english_tokenizer(txt) for txt in train_ds["text"])
vocab, unk_idx = build_vocab_from_iterator(token_stream)
pad_idx = vocab["<pad>"]

# ---------------------------------------------------------------
# 3) Encode & collate for DataLoader
# ---------------------------------------------------------------
def encode(text):
    return [vocab.get(tok, unk_idx) for tok in basic_english_tokenizer(text)]

def collate(batch):
    # batch: list of dicts { 'text': str, 'label': int }
    seqs   = [torch.tensor(encode(ex["text"]), dtype=torch.long) for ex in batch]
    labels = torch.tensor([ex["label"] for ex in batch], dtype=torch.long)
    lengths= torch.tensor([len(s) for s in seqs], dtype=torch.long)

    padded = pad_sequence(seqs, batch_first=True, padding_value=pad_idx)
    return padded, lengths, labels

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate)
val_loader   = DataLoader(test_ds,  batch_size=128, shuffle=False, collate_fn=collate)

# ---------------------------------------------------------------
# 4) Bi-LSTM model
# ---------------------------------------------------------------
class TextBiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=200, hidden=128,
                 n_layers=1, pad_idx=0, n_classes=2, dropout=0.3):
        super().__init__()
        self.embed   = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm    = nn.LSTM(embed_dim, hidden, num_layers=n_layers,
                               bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden*2, n_classes)

    def forward(self, x, lengths):
        x = self.embed(x)  # (B, T, D)
        packed = pack_padded_sequence(x, lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(out, batch_first=True)  # (B, T, 2H)

        # mean pooling over the valid timesteps
        mask = (torch.arange(out.size(1))[None, :].to(out.device)
                < lengths[:, None])
        rep = (out * mask.unsqueeze(-1)).sum(1) / lengths.unsqueeze(1)
        rep = self.dropout(rep)
        return self.fc(rep)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = TextBiLSTM(len(vocab), pad_idx=pad_idx).to(device)

# ---------------------------------------------------------------
# 5) Training setup
# ---------------------------------------------------------------
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def accuracy(net, loader):
    net.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, ln, yb in loader:
            xb, ln, yb = xb.to(device), ln.to(device), yb.to(device)
            preds = net(xb, ln).argmax(1)
            correct += (preds == yb).sum().item()
            total   += yb.size(0)
    return correct / total

# ---------------------------------------------------------------
# 6) Training loop
# ---------------------------------------------------------------
epochs = 5
for ep in range(1, epochs+1):
    model.train()
    pbar = tqdm(train_loader, desc=f"Epoch {ep}", leave=False)
    for xb, ln, yb in pbar:
        xb, ln, yb = xb.to(device), ln.to(device), yb.to(device)
        logits = model(xb, ln)
        loss   = criterion(logits, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        pbar.set_postfix(loss=f"{loss.item():.3f}")

    val_acc = accuracy(model, val_loader)
    print(f"Epoch {ep}: val accuracy {val_acc*100:.2f}%")

# ---------------------------------------------------------------
# 7) Save fine-tuned weights
# ---------------------------------------------------------------
torch.save(model.state_dict(), "bi_lstm_imdb.pt")
print("✅ Model saved to bi_lstm_imdb.pt")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

unsupervised-00000-of-00001.parquet:   0%|          | 0.00/42.0M [00:00<?, ?B/s]

...

Epoch 3: val accuracy 88.46%


Epoch 4:   0%|          | 0/391 [00:00<?, ?it/s]

Epoch 4: val accuracy 87.82%


Epoch 5:   0%|          | 0/391 [00:00<?, ?it/s]

Epoch 5: val accuracy 87.45%
✅ Model saved to bi_lstm_imdb.pt
